# Standard Model Walkthrough

This notebook uses `theories.build_standard_model()` to show the modern public surface of the project.

The split is deliberate:

- `feynpy` provides the reusable engine: `Model`, compiled Lagrangians, validation, vertex reporting, flavor expansion, and field transformations.
- `theories` provides concrete theory applications. Here that means the packaged Standard Model builder plus its source sectors, physical fields, parameters, and transformation rules.

The notebook stays on the public API. It does not reach into compiler internals.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from theories import build_standard_model


def signature_rows(signatures, limit=None):
    items = signatures if limit is None else signatures[:limit]
    rows = []
    for sig in items:
        rows.append(
            {
                "names": sig.names,
                "arity": sig.arity,
                "term_count": sig.term_count,
                "sectors": sig.sectors,
            }
        )
    return rows


def issue_rows(report):
    return [
        {
            "severity": issue.severity,
            "code": issue.code,
            "message": issue.message,
        }
        for issue in report.issues
    ]


def term_rows(terms, limit=5):
    rows = []
    for term in terms[:limit]:
        rows.append(
            {
                "label": term.label,
                "coupling": term.coupling,
                "field_count": len(term.fields),
                "derivative_count": len(term.derivatives),
                "sector": getattr(term, "sector", ""),
            }
        )
    return rows


def sector_counts(compiled):
    return {
        sector: len(compiled.vertex_signatures(sector=sector))
        for sector in ("pure_gauge", "matter", "ghost", "gauge_fixing")
    }


print("Python:", sys.executable)
print("Repository root:", ROOT)


## 1. Build the packaged Standard Model

`build_standard_model()` returns a `StandardModel` dataclass bundle. It contains:

- `source_model`: the gauge-basis source declaration,
- `model`: the broken-phase model object,
- `lagrangian`: the compiled broken-phase `CompiledLagrangian`,
- `fields`, `parameters`, `gauge_groups`, `lagrangians`, and `transformations`.

That is the application layer in `theories`: a concrete theory assembled out of generic `feynpy` machinery.

In [ ]:
sm = build_standard_model()
L = sm.lagrangian
fields = sm.fields
source_compiled = sm.source_model.lagrangian()

summary = {
    "source model name": sm.source_model.name,
    "broken model name": sm.model.name,
    "source gauge groups": [group.name for group in sm.source_model.gauge_groups],
    "source field count": len(sm.source_model.fields),
    "broken field count": len(sm.model.fields),
    "source compiled term count": len(source_compiled.terms),
    "broken compiled term count": len(L.terms),
    "broken compiled signature count": len(L.vertex_signatures()),
    "transformation count": len(sm.transformations),
    "source declared sector sizes": {
        name: len(getattr(sm.lagrangians, name).source_terms)
        for name in sm.lagrangians.__dataclass_fields__
    },
}

pprint(summary)


## 2. Source-basis content, broken-basis content, and the transformation stage

The `source_model` still knows about the gauge-basis fields and gauge groups (`B`, `Wi`, `Phi`, `QL`, ...). The broken bundle instead exposes the physical fields (`A`, `Z`, `W`, `H`, `G0`, `GP`, `u`, `d`, ...).

The transformation rules stored on `sm.transformations` are part of the general engine capability, but here they are packaged as the Standard Model application of that capability.

In [ ]:
print("source gauge-basis field names:")
print([field.name for field in sm.source_model.fields])
print()
print("first broken-basis field names:")
print([field.name for field in sm.model.fields[:16]])
print()
print("sample transformation rules:")
pprint(
    [
        {
            "source": rule.source.name,
            "components": rule.components,
            "uses_builder": rule.builder is not None,
            "term_count": len(rule.terms),
        }
        for rule in sm.transformations[:8]
    ]
)
print()
print("sample source signatures:")
pprint(signature_rows(source_compiled.vertex_signatures(), limit=10))
print()
print("sample broken 3-point signatures:")
pprint(signature_rows(L.vertex_signatures(arity=3), limit=12))


## 3. Validation

The same public validation API works on both the declared model and the compiled Lagrangian.

In [ ]:
source_report = sm.source_model.validate()
broken_model_report = sm.model.validate()
broken_lagrangian_report = L.validate()

for label, report in (
    ("source model", source_report),
    ("broken model", broken_model_report),
    ("broken compiled lagrangian", broken_lagrangian_report),
):
    print(label)
    print({
        "ok": report.ok,
        "issue_count": len(report.issues),
        "error_count": len(report.errors),
        "warning_count": len(report.warnings),
    })
    if report.issues:
        pprint(issue_rows(report))
    print()


## 4. Packaged theory metadata

This is where the `theories` layer is useful: it gives a ready-made theory bundle with named physical fields, packaged flavor classes, and predefined parameters.

In [ ]:
e, mu, ta = fields.l.class_members
u, c, t = fields.uq.class_members
d, s, b = fields.dq.class_members

metadata_summary = {
    "source lookup": {
        "find gauge group SU2L": sm.source_model.find_gauge_group("SU2L").name,
        "find gauge-basis field B": sm.source_model.find_field("B").name,
        "find parameter CKM": sm.source_model.find_parameter("CKM").name,
    },
    "broken lookup": {
        "find field A": sm.model.find_field("A").name,
        "find field W": sm.model.find_field("W").name,
    },
    "physical masses": {
        "A": fields.A.mass,
        "W": fields.W.mass,
        "Z": fields.Z.mass,
        "H": fields.H.mass,
    },
    "derived field relations": {
        "G0 goldstone_of": fields.G0.goldstone_of.name,
        "GP goldstone_of": fields.GP.goldstone_of.name,
        "ghWp ghost_of": fields.ghWp.ghost_of.name,
        "ghZ ghost_of": fields.ghZ.ghost_of.name,
    },
    "flavor classes": {
        "charged leptons": [member.name for member in fields.l.class_members],
        "up quarks": [member.name for member in fields.uq.class_members],
        "down quarks": [member.name for member in fields.dq.class_members],
    },
    "parameter sample": {
        "sw value": sm.parameters.sw.value,
        "cw value": sm.parameters.cw.value,
        "MW value": sm.parameters.MW.value,
        "CKM(1,1)": sm.parameters.CKM.components[(1, 1)],
        "CKM(1,2)": sm.parameters.CKM.components[(1, 2)],
    },
}

pprint(metadata_summary)


## 5. Vertex reporting and filtering

The broken compiled Lagrangian supports grouped signature reporting before you extract any explicit vertex formula.

In [ ]:
report = L.vertex_report()

print("global report summary:")
pprint(
    {
        "total_terms": report.total_terms,
        "total_signatures": report.total_signatures,
        "matched_terms": report.matched_terms,
        "matched_signatures": report.matched_signatures,
        "sector_counts": sector_counts(L),
    }
)
print()
print("pure-gauge signatures:")
pprint(signature_rows(L.vertex_signatures(sector="pure_gauge")))
print()
print("gauge-fixing signatures (first 5):")
pprint(signature_rows(L.vertex_signatures(sector="gauge_fixing"), limit=5))
print()
print("exact H W- W+ signature:")
pprint(signature_rows(L.vertex_signatures(signature=(fields.H, fields.W.bar, fields.W))))
print()
print("signatures containing W- and W+ (first 10):")
pprint(signature_rows(L.vertex_signatures(contains_fields=(fields.W.bar, fields.W)), limit=10))


## 6. Inspect the exact compiled terms behind one signature

`matching_terms(...)` lets you look at the raw compiled monomials that later get aggregated into one vertex signature. This is especially useful for large gauge vertices.

In [ ]:
hww_terms = L.matching_terms(fields.H, fields.W.bar, fields.W)
wwa_terms = L.matching_terms(fields.W.bar, fields.W, fields.A)
ghost_terms = L.matching_terms(fields.ghWp.bar, fields.ghWp, sector="ghost")

pprint(
    {
        "H W- W+ raw term count": len(hww_terms),
        "W- W+ A raw term count": len(wwa_terms),
        "ghost W bilinear raw term count": len(ghost_terms),
    }
)
print()
print("representative H W- W+ term(s):")
pprint(term_rows(hww_terms))
print()
print("representative W- W+ A term(s):")
pprint(term_rows(wwa_terms, limit=3))
print()
print("representative ghost terms:")
pprint(term_rows(ghost_terms, limit=3))


## 7. Representative Feynman rules

The same compiled object can then produce explicit vertex factors.

In [ ]:
representative_rules = {
    "Gamma(W-, W+, A)": L.feynman_rule(fields.W.bar, fields.W, fields.A, simplify=True),
    "Gamma(H, W-, W+)": L.feynman_rule(fields.H, fields.W.bar, fields.W, simplify=True),
    "Gamma(ghW+bar, ghW+)": L.feynman_rule(fields.ghWp.bar, fields.ghWp, simplify=True),
}

for title, expression in representative_rules.items():
    print(title)
    print(expression)
    print()


## 8. Zero-argument extraction for a selected set of signatures

`feynman_rule(select=[...])` returns a dictionary of grouped vertices. This is convenient when you want a small report rather than a single explicit call.

In [ ]:
selected_rules = L.feynman_rule(
    select=[
        (fields.W.bar, fields.W, fields.A),
        (fields.H, fields.W.bar, fields.W),
        (fields.ghG.bar, fields.ghG, fields.G),
    ],
    simplify=True,
)

print("selected signatures:")
pprint(list(selected_rules.keys()))
print()
for signature, expression in selected_rules.items():
    print(signature)
    print(expression)
    print()


## 9. Flavor expansion on the packaged Standard Model classes

The Standard Model bundle keeps flavor classes in compact form by default (`l`, `uq`, `dq`). The same compiled object can be queried in the flavor-expanded view.

In [ ]:
print("compact lepton-photon signature:")
pprint(signature_rows(L.vertex_signatures(signature=(fields.l.bar, fields.l, fields.A))))
print()
print("expanded electron-photon signature:")
pprint(signature_rows(L.vertex_signatures(signature=(e.bar, e, fields.A), flavor_expand=True)))
print()
print("Gamma(ebar, e, A):")
print(L.feynman_rule(e.bar, e, fields.A, flavor_expand=True, simplify=True))
print()
print("Gamma(ubar, d, W+):")
print(L.feynman_rule(u.bar, d, fields.W, flavor_expand=True, simplify=True))
print()
print("Off-diagonal mu.bar, e, A is absent:")
try:
    print(L.feynman_rule(mu.bar, e, fields.A, flavor_expand=True, simplify=True))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


## 10. Output-policy options

The high-level extraction API defaults to stripped external legs and omits the universal momentum-conservation delta. Both behaviors are configurable.

In [ ]:
compact = L.feynman_rule(e.bar, e, fields.A, flavor_expand=True, simplify=True)
full = L.feynman_rule(
    e.bar,
    e,
    fields.A,
    flavor_expand=True,
    simplify=True,
    include_delta=True,
    strip_externals=False,
)

print("default photon vertex contains Delta(...)?:", "Delta(" in compact.to_canonical_string())
print("full photon vertex contains Delta(...)?:", "Delta(" in full.to_canonical_string())
print("default photon vertex contains external U(...)?", "U(" in compact.to_canonical_string())
print("full photon vertex contains external U(...)?", "U(" in full.to_canonical_string())


## 11. Builder variants

The theory application also exposes a few configuration switches at construction time. The generic engine stays the same; only the packaged theory content changes.

In [ ]:
variant_rows = []
for kwargs in (
    {},
    {"include_ghosts": False},
    {"include_gauge_fixing": False},
    {"include_ghosts": False, "include_gauge_fixing": False},
):
    variant = build_standard_model(**kwargs)
    variant_rows.append(
        {
            "options": kwargs or {"default": True},
            "term_count": len(variant.lagrangian.terms),
            "signature_count": len(variant.lagrangian.vertex_signatures()),
            "sector_counts": sector_counts(variant.lagrangian),
        }
    )

pprint(variant_rows)


## 12. Takeaway

This notebook uses one concrete `theories` package, but every capability it exercised is a generic `feynpy` surface:

- model validation,
- compiled-lagrangian validation,
- grouped signature reporting,
- exact-term matching,
- vertex extraction,
- flavor expansion,
- configurable output policy,
- theory construction by a packaged field-transformation stage.

What is Standard-Model-specific here is only the packaged content: the named fields, parameters, source sectors, and transformation rules returned by `build_standard_model()`.